# Auto-tuning of GPU code

An important aspect of GPU programming is choosing the optimal thread block size. If there are too many or too few threads per block, performance will not be optimal. For the block size, it is common to try a few different options and see how they affect the speed of your code to determine what works best.

However, this is a common occurrence in GPU programming. Sometimes we have parameters that have no effect on the output of the code (i.e., the program always computes the correct results), but the performance can vary significantly. We call these types of parameters *tunable parameters*.

Sometimes there are many tunable parameters. For example, there can be five or even ten parameters. In these cases, it is best to use an auto-tuner to automatically try many combinations of parameters and find the optimal configuration.

In this lesson, we will experiment with such an auto-tuner called **Kernel Tuner**. Kernel Tuner is an open-source Python library that can be used to tune GPU kernels from many languages (including CUDA). You can read more about Kernel Tuner here: https://kerneltuner.github.io

In [2]:
import kernel_tuner

/home/sheldens/.local/lib/python3.13/site-packages/cupy/_environment.py:663: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''
/home/sheldens/.local/lib/python3.13/site-packages/kernel_tuner/observers/nvml.py:10: FutureWarning: The pynvml package i

Below is a basic an example of how to use Kernel Tuner. The GPU kernel that we will tune is a matrix multiplication, a common operation in GPU computing. 

Using Kernel Tuner involves three main steps:

* Generate some input data.
* Provide the kernel code as a string.
* Define the tuning options and call Kernel Tuner.

Kernel Tuner has many options that you can pass to `tune_kernel`. We will focus on the most important ones. You can read more about it here:  https://kerneltuner.github.io/kernel_tuner/stable/user-api.html

In [3]:
import numpy as np
import kernel_tuner as kt

# Input data
n = 2048
A = np.random.rand(n, n).astype("float32")
B = np.random.rand(n, n).astype("float32")
C_expected = np.matmul(A, B)
C = np.zeros_like(C_expected)

# Kernel code
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    for (int k = 0; k < n; k++) {
      // C[i,j] += A[i,k] * B[k,j]
      sum += A[i * n + k] * B[k * n + j];
    }
    C[i * n + j] += sum;
  }
}
"""

# Tuning options
problem_size = (n, n)
arguments = [np.int32(n), A, B, C]
answer = [None, None, None, C_expected]

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]

# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, time=18.786ms
block_size_x=16, time=9.734ms
block_size_x=32, time=6.406ms
block_size_x=64, time=6.250ms
block_size_x=128, time=6.293ms
best performing configuration:
block_size_x=64, time=6.250ms


# Exercise A: Tuning the block size
Extend the tuner by adding a `block_size_y` parameter. Experiment with different values and determine which configuration gives the best performance.

In [4]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    for (int k = 0; k < n; k++) {
      sum += A[i * n + k] * B[k * n + j];
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8, 16]

# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, block_size_y=1, time=18.797ms
block_size_x=8, block_size_y=2, time=11.285ms
block_size_x=8, block_size_y=4, time=9.009ms
block_size_x=8, block_size_y=8, time=8.916ms
block_size_x=8, block_size_y=16, time=8.921ms
block_size_x=16, block_size_y=1, time=9.740ms
block_size_x=16, block_size_y=2, time=5.824ms
block_size_x=16, block_size_y=4, time=5.444ms
block_size_x=16, block_size_y=8, time=5.457ms
block_size_x=16, block_size_y=16, time=5.661ms
block_size_x=32, block_size_y=1, time=6.404ms
block_size_x=32, block_size_y=2, time=4.820ms
block_size_x=32, block_size_y=4, time=4.819ms
block_size_x=32, block_size_y=8, time=4.864ms
block_size_x=32, block_size_y=16, time=4.956ms
block_size_x=64, block_size_y=1, time=6.254ms
block_size_x=64, block_size_y=2, time=4.792ms
block_size_x=64, block_size_y=4, time=4.813ms
block_size_x=64, block_size_y=8, time=4.966ms
block_size_x=64, block_size_y=16, time=4.765ms
block_size_x=128, block_size_y=1, time=6.289ms
blo

As you may have noticed, performance becomes very poor if the total number of threads per block (which is equal to block_size_x * block_size_y) becomes very small.

We can add a so-called *restriction* to Kernel Tuner. Restrictions are strings that describe Boolean conditions. Kernel Tuner will skip configurations for which any of these restrictions evaluates to False. This allows you to exclude combinations of parameters that are known not to work well. In the example below, see how this works. Try to add another restriction that involves `block_size_x` and `block_size_y`.

In [5]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    for (int k = 0; k < n; k++) {
      sum += A[i * n + k] * B[k * n + j];
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8, 16]

restrictions = [
    "block_size_x * block_size_y >= 64",
]
        
# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
    restrictions=restrictions,
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, block_size_y=8, time=8.915ms
block_size_x=8, block_size_y=16, time=8.921ms
block_size_x=16, block_size_y=4, time=5.447ms
block_size_x=16, block_size_y=8, time=5.455ms
block_size_x=16, block_size_y=16, time=5.657ms
block_size_x=32, block_size_y=2, time=4.780ms
block_size_x=32, block_size_y=4, time=4.824ms
block_size_x=32, block_size_y=8, time=4.861ms
block_size_x=32, block_size_y=16, time=4.960ms
block_size_x=64, block_size_y=1, time=6.257ms
block_size_x=64, block_size_y=2, time=4.798ms
block_size_x=64, block_size_y=4, time=4.813ms
block_size_x=64, block_size_y=8, time=4.967ms
block_size_x=64, block_size_y=16, time=4.766ms
block_size_x=128, block_size_y=1, time=6.288ms
block_size_x=128, block_size_y=2, time=4.736ms
block_size_x=128, block_size_y=4, time=4.844ms
block_size_x=128, block_size_y=8, time=4.655ms
best performing configuration:
block_size_x=128, block_size_y=8, time=4.655ms


In [6]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    for (int k = 0; k < n; k++) {
      sum += A[i * n + k] * B[k * n + j];
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8, 16]

restrictions = [
    "block_size_x * block_size_y >= 64",
]
        
# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
    restrictions=restrictions,
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, block_size_y=8, time=8.918ms
block_size_x=8, block_size_y=16, time=8.926ms
block_size_x=16, block_size_y=4, time=5.460ms
block_size_x=16, block_size_y=8, time=5.476ms
block_size_x=16, block_size_y=16, time=5.681ms
block_size_x=32, block_size_y=2, time=4.806ms
block_size_x=32, block_size_y=4, time=4.817ms
block_size_x=32, block_size_y=8, time=4.860ms
block_size_x=32, block_size_y=16, time=4.952ms
block_size_x=64, block_size_y=1, time=6.255ms
block_size_x=64, block_size_y=2, time=4.797ms
block_size_x=64, block_size_y=4, time=4.807ms
block_size_x=64, block_size_y=8, time=4.953ms
block_size_x=64, block_size_y=16, time=4.769ms
block_size_x=128, block_size_y=1, time=6.291ms
block_size_x=128, block_size_y=2, time=4.743ms
block_size_x=128, block_size_y=4, time=4.831ms
block_size_x=128, block_size_y=8, time=4.656ms
best performing configuration:
block_size_x=128, block_size_y=8, time=4.656ms


# Exercise B: Unrolling loops

Next, we can introduce another tunable parameter. In addition to adjusting the block size, we can also experiment with different compiler directives that may affect performance.

One example is the CUDA annotation `#pragma unroll`, which instructs the compiler to *unroll* a loop. For example, if we would use `#pragma unroll 5`, then CUDA would essentially replace our loop by:

```
for (int k = 0; k < n; k += 5) {
    sum += A[i * n + (k+0)] * B[(k+0) * n + j];
    sum += A[i * n + (k+1)] * B[(k+1) * n + j];
    sum += A[i * n + (k+2)] * B[(k+2) * n + j];
    sum += A[i * n + (k+3)] * B[(k+3) * n + j];
    sum += A[i * n + (k+4)] * B[(k+4) * n + j];
}
```

Loop unrolling can reduce loop overhead and sometimes improve performance by introducing more instruction-level parallelism. However, unrolling a loop may also increase register usage or code size, which means that the optimal choice can depend on the specific GPU and kernel.

We can treat the loop unrolling factor as a tunable parameter and let the auto-tuner explore different options to determine which configuration gives the best performance.

In [7]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
      sum += A[i * n + k] * B[k * n + j];
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8]
tune_params["loop_unroll_factor"] = [1, 5, 25, 100]

# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, block_size_y=1, loop_unroll_factor=1, time=52.657ms
block_size_x=8, block_size_y=1, loop_unroll_factor=5, time=21.459ms
block_size_x=8, block_size_y=1, loop_unroll_factor=25, time=18.846ms
block_size_x=8, block_size_y=1, loop_unroll_factor=100, time=18.599ms
block_size_x=8, block_size_y=2, loop_unroll_factor=1, time=26.999ms
block_size_x=8, block_size_y=2, loop_unroll_factor=5, time=12.009ms
block_size_x=8, block_size_y=2, loop_unroll_factor=25, time=11.004ms
block_size_x=8, block_size_y=2, loop_unroll_factor=100, time=11.109ms
block_size_x=8, block_size_y=4, loop_unroll_factor=1, time=13.997ms
block_size_x=8, block_size_y=4, loop_unroll_factor=5, time=8.982ms
block_size_x=8, block_size_y=4, loop_unroll_factor=25, time=8.935ms
block_size_x=8, block_size_y=4, loop_unroll_factor=100, time=8.963ms
block_size_x=8, block_size_y=8, loop_unroll_factor=1, time=9.599ms
block_size_x=8, block_size_y=8, loop_unroll_factor=5, time=8.913ms
block_size_x=8,

# Exercise C: Using L1 cache reads

Let's now introduce yet another tunable parameter.

In CUDA, we can use the special LDG instruction to perform a read-only L1/Tex cache load: [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/cpp-language-extensions.html#low-level-load-and-store-functions)

Letting reads go through the L1/Tex cache may sometimes speed up your GPU kernel because it allows the GPU to put frequently accessed data in the cache. This is especially beneficial when multiple threads read the same data or when threads read data close to each other (this is called *spatial locality*) or at almost the same time (this is called *temporal locality*).

However, in other cases it may not improve performance. For example, when the compiler already applies this optimization for kernel. In some cases it may even slow the kernel if the additional overhead of using `__ldg` may outweigh its benefits.

We can add a tunable parameter (call it `use_ldg`) that controls if we disable LDG (`use_ldg==0`) or enable `__ldg` (`use_ldg==0`).

Hint: You can replace `A[i * n + k]` by `__ldg(&A[i * n + k])` to use the LDG instruction.

In [8]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
#if use_ldg
      sum += __ldg(&A[i * n + k]) * __ldg(&B[k * n + j]);
#else
      sum += A[i * n + k] * B[k * n + j];
#endif
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [32, 64]
tune_params["block_size_y"] = [4, 8]
tune_params["loop_unroll_factor"] = [1, 10]
tune_params["use_ldg"] = [0, 1]

# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_ldg=0, time=7.488ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_ldg=1, time=7.793ms
block_size_x=32, block_size_y=4, loop_unroll_factor=10, use_ldg=0, time=4.749ms
block_size_x=32, block_size_y=4, loop_unroll_factor=10, use_ldg=1, time=5.681ms
block_size_x=32, block_size_y=8, loop_unroll_factor=1, use_ldg=0, time=7.374ms
block_size_x=32, block_size_y=8, loop_unroll_factor=1, use_ldg=1, time=7.653ms
block_size_x=32, block_size_y=8, loop_unroll_factor=10, use_ldg=0, time=4.735ms
block_size_x=32, block_size_y=8, loop_unroll_factor=10, use_ldg=1, time=5.618ms
block_size_x=64, block_size_y=4, loop_unroll_factor=1, use_ldg=0, time=7.474ms
block_size_x=64, block_size_y=4, loop_unroll_factor=1, use_ldg=1, time=7.785ms
block_size_x=64, block_size_y=4, loop_unroll_factor=10, use_ldg=0, time=4.714ms
block_size_x=64, block_size_y=4, loop_unroll_factor=10, use_ldg=1, time=5.730ms
block_size_x=64, 

# Exercise D: Using FMA

Our kernel contains a line that looks like:

```
sum += x * y;
```

This requires the GPU to perform two operations: `x * y` to multiply the two operands, and `sum += result` to add the result to `sum`.

However, CUDA provides a special function that can perform these two operations in a single instruction. This intrinsic is called `fmaf`, which stands for ***f**used **m**ultiply-**a**dd* on **f**loats. An *intrinsic* is a special function provided by the compiler that maps directly to a hardware instruction.

To use this intrinsic, you can replace a line such as:

```
sum += a * b;
```

with

```
sum = fmaf(a, b, sum);
```


Add a new tunable parameter to your kernel called `use_fma`. If it is set to `1`, use the `fmaf` intrinsic. If it is `0`, use the regular operations.


In [9]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    for (int k = 0; k < n; k++) {
#if use_fma
      sum = fmaf(A[i * n + k], B[k * n + j], sum);
#else
      sum += A[i * n + k] * B[k * n + j];
#endif
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8, 16]
tune_params["use_fmaf"] = [0, 1]

restrictions = [
    "block_size_x * block_size_y >= 64",
]
        
# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="cuda",
    restrictions=restrictions,
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, block_size_y=8, use_fmaf=0, time=8.913ms
block_size_x=8, block_size_y=8, use_fmaf=1, time=8.919ms
block_size_x=8, block_size_y=16, use_fmaf=0, time=8.921ms
block_size_x=8, block_size_y=16, use_fmaf=1, time=8.921ms
block_size_x=16, block_size_y=4, use_fmaf=0, time=5.445ms
block_size_x=16, block_size_y=4, use_fmaf=1, time=5.444ms
block_size_x=16, block_size_y=8, use_fmaf=0, time=5.457ms
block_size_x=16, block_size_y=8, use_fmaf=1, time=5.457ms
block_size_x=16, block_size_y=16, use_fmaf=0, time=5.664ms
block_size_x=16, block_size_y=16, use_fmaf=1, time=5.658ms
block_size_x=32, block_size_y=2, use_fmaf=0, time=4.769ms
block_size_x=32, block_size_y=2, use_fmaf=1, time=4.767ms
block_size_x=32, block_size_y=4, use_fmaf=0, time=4.823ms
block_size_x=32, block_size_y=4, use_fmaf=1, time=4.822ms
block_size_x=32, block_size_y=8, use_fmaf=0, time=4.863ms
block_size_x=32, block_size_y=8, use_fmaf=1, time=4.863ms
block_size_x=32, block_size_y=16, use_fmaf=

# Exercise E

We can also use the `matmul` function from `cupy` to perform the matrix multiplication. Compare its performance to that of your tuned kernel. How close can you get to the performance of CuPy’s implementation, and which configuration gives the best result?

Why do you think that is cupy is faster? Think of reasons of the CuPy team did to achieve this performance level.

In [10]:
import cupy

A_gpu = cupy.array(A)
B_gpu = cupy.array(B)
C_gpu = cupy.matmul(A_gpu, B_gpu)

%timeit cupy.matmul(A_gpu, B_gpu); cupy.cuda.stream.get_current_stream().synchronize()

1 ms ± 122 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


# Exercise F: hard! (optional)

Currently, each thread computes one entry of the output matrix `C[i,j]`. Try to modify your kernel such that each thread calculates two entries in the output matrix (for example, `C[2*i, j]` and `C[2*i+1, j]`). This does this improve performance? Why does/doesn't it do you think?

In [15]:
kernel_name = "matmul"
kernel_code = """
__global__ void __launch_bounds__(block_size_x*block_size_y) matmul(int n, const float* __restrict__ A, const float* __restrict__ B, float* __restrict__ C) {
  int j_start = vector_size*(blockIdx.x * tile_size_x * blockDim.x + threadIdx.x);
  int i_start = tile_size_y*(blockIdx.y * blockDim.y + threadIdx.y);

  if (i_start < n && j_start < n) {
    float sum[tile_size_y][tile_size_x][vector_size];

    #pragma unroll
    for (int i_offset = 0; i_offset < tile_size_y; i_offset++) {
        #pragma unroll
        for (int j_offset = 0; j_offset < tile_size_x; j_offset++) {
            #pragma unroll
            for (int u = 0; u < vector_size; u++) {
                sum[i_offset][j_offset][u] = 0;
            }
        }
    }
    
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
        float local_A[tile_size_y];
        float local_B[tile_size_x][vector_size];
        
        #pragma unroll
        for (int i_offset = 0; i_offset < tile_size_y; i_offset++) {
            int i = i_start + i_offset;
            local_A[i_offset] = use_ldg ? __ldg(&A[i * n + k]) : A[i * n + k];
        }

        #pragma unroll
        for (int j_offset = 0; j_offset < tile_size_x; j_offset++) {
            const float* B_ptr = &B[k * n + j_start + j_offset*block_size_x*vector_size];

#if vector_size == 1        
            float value = use_ldg ? __ldg(B_ptr) : *B_ptr;
            local_B[j_offset][0] = value;
#elif vector_size == 2
            float2 v = use_ldg ? __ldg(reinterpret_cast<const float2*>(B_ptr)) : *reinterpret_cast<const float2*>(B_ptr);
            local_B[j_offset][0] = v.x;
            local_B[j_offset][1] = v.y;
#elif vector_size == 4
            float4 v = use_ldg ? __ldg(reinterpret_cast<const float4*>(B_ptr)) : *reinterpret_cast<const float4*>(B_ptr);
            local_B[j_offset][0] = v.x;
            local_B[j_offset][1] = v.y;
            local_B[j_offset][2] = v.z;
            local_B[j_offset][3] = v.w;
#elif vector_size == 8
            float4 v0 = use_ldg ? __ldg(reinterpret_cast<const float4*>(B_ptr)) : *reinterpret_cast<const float4*>(B_ptr);
            local_B[j_offset][0] = v0.x;
            local_B[j_offset][1] = v0.y;
            local_B[j_offset][2] = v0.z;
            local_B[j_offset][3] = v0.w;
            float4 v1 = use_ldg ? __ldg(reinterpret_cast<const float4*>(B_ptr + 4)) : *reinterpret_cast<const float4*>(B_ptr + 4);
            local_B[j_offset][4] = v1.x;
            local_B[j_offset][5] = v1.y;
            local_B[j_offset][6] = v1.z;
            local_B[j_offset][7] = v1.w;
#endif
        }
    
        #pragma unroll
        for (int i_offset = 0; i_offset < tile_size_y; i_offset++) {
            #pragma unroll
            for (int j_offset = 0; j_offset < tile_size_x; j_offset++) {
                #pragma unroll
                for (int lane = 0; lane < vector_size; lane++) {
                    sum[i_offset][j_offset][lane] += local_A[i_offset] * local_B[j_offset][lane];
                }
            }
        }
    }
    #pragma unroll
    for (int i_offset = 0; i_offset < tile_size_y; i_offset++) {
        #pragma unroll
        for (int j_offset = 0; j_offset < tile_size_x; j_offset++) {
            for (int lane = 0; lane < vector_size; lane++) {
                int i = i_start + i_offset;
                int j = j_start + j_offset*block_size_x*vector_size + lane;
                C[i * n + j] += sum[i_offset][j_offset][lane];
            }
        }
    }
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8, 16, 32]
tune_params["loop_unroll_factor"] = [1, 10, 25]
tune_params["use_ldg"] = [0, 1]
tune_params["tile_size_x"] = [1, 2, 4, 8]
tune_params["tile_size_y"] = [1, 2, 4, 8]
tune_params["vector_size"] = [1, 2, 4, 8]

restrictions = [
    f"{n} % (block_size_x * tile_size_x * vector_size) == 0",
    f"{n} % (block_size_y * tile_size_y) == 0"
]

# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    restrictions=restrictions,
    strategy="genetic_algorithm",
    lang="cuda",
);

Using: NVIDIA A100-SXM4-40GB
block_size_x=8, block_size_y=32, loop_unroll_factor=25, use_ldg=0, tile_size_x=1, tile_size_y=2, vector_size=1, time=8.059ms
block_size_x=64, block_size_y=1, loop_unroll_factor=10, use_ldg=0, tile_size_x=2, tile_size_y=4, vector_size=8, time=2.185ms
block_size_x=8, block_size_y=2, loop_unroll_factor=1, use_ldg=1, tile_size_x=2, tile_size_y=1, vector_size=1, time=15.493ms
block_size_x=8, block_size_y=1, loop_unroll_factor=10, use_ldg=1, tile_size_x=1, tile_size_y=1, vector_size=1, time=21.470ms
block_size_x=32, block_size_y=4, loop_unroll_factor=25, use_ldg=1, tile_size_x=1, tile_size_y=2, vector_size=4, time=2.171ms
block_size_x=16, block_size_y=1, loop_unroll_factor=25, use_ldg=0, tile_size_x=4, tile_size_y=4, vector_size=2, time=2.989ms
block_size_x=32, block_size_y=16, loop_unroll_factor=1, use_ldg=1, tile_size_x=1, tile_size_y=2, vector_size=4, time=2.668ms
block_size_x=32, block_size_y=1, loop_unroll_factor=10, use_ldg=1, tile_size_x=4, tile_size_y=2, 